<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 从零开始的字节对编码（BPE）Tokenizer —— 简单版

- 这是一个独立的笔记本，出于教育目的从零实现了流行的字节对编码（BPE）tokenization 算法，该算法用于 GPT-2 到 GPT-4、Llama 3 等模型中
- 有关 tokenization 目的的更多详细信息，请参阅[第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)；此处的代码是解释 BPE 算法的补充材料
- OpenAI 为训练原始 GPT 模型而实现的原始 BPE tokenizer 可以在[这里](https://github.com/openai/gpt-2/blob/master/src/encoder.py)找到
- BPE 算法最初于 1994 年描述：Philip Gage 的"[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)"
- 如今大多数项目（包括 Llama 3）都使用 OpenAI 的开源 [tiktoken 库](https://github.com/openai/tiktoken)，因为它的计算性能更好；它允许加载预训练的 GPT-2 和 GPT-4 tokenizer（例如，Llama 3 模型也是使用 GPT-4 tokenizer 训练的）
- 上述实现与本笔记本中我的实现之间的区别在于，除了其他方面外，我的实现还包含一个用于训练 tokenizer 的函数（用于教育目的）
- 还有一个名为 [minBPE](https://github.com/karpathy/minbpe) 的实现支持训练，可能性能更好（我在这里的实现专注于教育目的）；与 `minbpe` 相比，我的实现还允许加载原始的 OpenAI tokenizer 词汇表和合并规则

**这是一个用于教育目的的非常朴素的实现。[bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) 笔记本包含了一个更复杂（但更难阅读）的实现，它与 tiktoken 的行为匹配。**

&nbsp;
# 1. 字节对编码（BPE）背后的主要思想

- BPE 的主要思想是将文本转换为整数表示（token ID），用于 LLM 训练（参见[第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="600px">

**图 2.6**：BPE tokenization 的整体流程。文本首先被分解为字节，然后通过迭代合并最频繁的相邻字节对来构建词汇表，最终将文本编码为 token ID 序列。

```
原始文本 ──▶ 分解为字节 ──▶ 迭代合并频繁对 ──▶ Token ID 序列
 "Hello"      [72,101,        合并 "ll"       [xxx, xxx]
               108,108,111]   合并 "He" ...
```

> **核心结论**：BPE 通过逐步合并高频字节对来构建词汇表，从而将文本压缩为更短的 token ID 序列。

&nbsp;
## 1.1 比特和字节

- 在进入 BPE 算法之前，让我们先介绍字节的概念
- 考虑将文本转换为字节数组（BPE 毕竟代表"字节"对编码）：

In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


- 当我们对 `bytearray` 对象调用 `list()` 时，每个字节都被视为一个单独的元素，结果是一个与字节值对应的整数列表：

In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- 这将是一种有效的将文本转换为 LLM embedding 层所需的 token ID 表示的方法
- 然而，这种方法的缺点是它为每个字符创建一个 ID（对于短文本来说这是很多 ID！）
- 也就是说，这意味着对于一个 17 个字符的输入文本，我们必须使用 17 个 token ID 作为 LLM 的输入：

In [3]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- 如果你之前使用过 LLM，你可能知道 BPE tokenizer 有一个词汇表，其中我们有整个单词或子词的 token ID，而不是每个字符一个
- 例如，GPT-2 tokenizer 将相同的文本（"This is some text"）仅编码为 4 个而不是 17 个 token：`1212, 318, 617, 2420`
- 你可以使用交互式 [tiktoken 应用](https://tiktokenizer.vercel.app/?model=gpt2)或 [tiktoken 库](https://github.com/openai/tiktoken)来验证：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```

**图 2.7**：tiktoken 交互式应用展示了 GPT-2 tokenizer 如何将文本编码为 token ID。注意每个单词被映射为单个 token，显著减少了序列长度。

| 编码方式 | "This is some text" 的 token 数 | 说明 |
|---------|-------------------------------|------|
| 字符级编码 | 17 | 每个字符一个 token |
| BPE (GPT-2) | 4 | 每个单词一个 token |

> **核心结论**：BPE 通过将完整单词映射为单个 token，大幅减少了 token 序列长度，提高了 LLM 的处理效率。

- 由于一个字节由 8 个比特组成，因此一个字节可以表示 2<sup>8</sup> = 256 个可能的值，范围从 0 到 255
- 你可以通过执行代码 `bytearray(range(0, 257))` 来确认这一点，它会警告你 `ValueError: byte must be in range(0, 256)`
- BPE tokenizer 通常使用这 256 个值作为其前 256 个单字符 token；可以通过运行以下代码来直观地检查：

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")


prints:
0: !
1: "
2: #
...
255:    # <---- 单字符 token 到这里为止
256:  t
257:  a
...
298: ent
299:  n

```

- 在上面，请注意条目 256 和 257 不是单字符值，而是双字符值（一个空格 + 一个字母），这是原始 GPT-2 BPE Tokenizer 的一个小缺点（这在 GPT-4 tokenizer 中已得到改进）

&nbsp;
## 1.2 构建词汇表

- BPE tokenization 算法的目标是构建一个包含常见子词的词汇表，如 `298: ent`（例如可以在 *entangle, entertain, enter, entrance, entity, ...* 中找到），甚至是完整的单词，如

```
318: is
617: some
1212: This
2420: text
```

- BPE 算法最初于 1994 年描述：Philip Gage 的"[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)"
- 在我们进入实际代码实现之前，当今用于 LLM tokenizer 的形式可以总结如下：

&nbsp;
## 1.3 BPE 算法概述

**1. 识别频繁对**
- 在每次迭代中，扫描文本以找到最常见的字节（或字符）对

**2. 替换并记录**

- 用一个新的占位符 ID 替换该对（一个尚未使用的 ID，例如，如果我们从 0...255 开始，第一个占位符将是 256）
- 将此映射记录在查找表中
- 查找表的大小是一个超参数，也称为"词汇表大小"（对于 GPT-2，这是 50,257）

**3. 重复直到没有增益**

- 继续重复步骤 1 和 2，不断合并最频繁的对
- 当无法进一步压缩时停止（例如，没有对出现超过一次）

**解压缩（解码）**

- 要恢复原始文本，通过用查找表中相应的对替换每个 ID 来反转该过程

&nbsp;
## 1.4 BPE 算法示例

### 1.4.1 编码部分的具体示例（步骤 1 和 2）

- 假设我们有文本（训练数据集）`the cat in the hat`，我们想从中为 BPE tokenizer 构建词汇表

**迭代 1**

1. 识别频繁对
  - 在此文本中，"th" 出现了两次（在开头和第二个 "e" 之前）

2. 替换并记录
  - 将 "th" 替换为一个新的、尚未使用的 token ID，例如 256
  - 新文本为：`<256>e cat in <256>e hat`
  - 新词汇表为

```
  0: ...
  ...
  256: "th"
```

**迭代 2**

1. **识别频繁对**  
   - 在文本 `<256>e cat in <256>e hat` 中，对 `<256>e` 出现了两次

2. **替换并记录**  
   - 将 `<256>e` 替换为一个新的、尚未使用的 token ID，例如 `257`
   - 新文本为：
     ```
     <257> cat in <257> hat
     ```
   - 更新后的词汇表为：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**迭代 3**

1. **识别频繁对**  
   - 在文本 `<257> cat in <257> hat` 中，对 `<257> ` 出现了两次（一次在开头，一次在 "hat" 之前）。

2. **替换并记录**  
   - 将 `<257> ` 替换为一个新的、尚未使用的 token ID，例如 `258`
   - 新文本为：
     ```
     <258>cat in <258>hat
     ```
   - 更新后的词汇表为：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- 以此类推

&nbsp;
### 1.4.2 解码部分的具体示例（步骤 3）

- 要恢复原始文本，我们按照引入的逆序将每个 token ID 替换为相应的对来反转该过程
- 从最终压缩文本开始：`<258>cat in <258>hat`
- 替换 `<258>` → `<257> `：`<257> cat in <257> hat`  
- 替换 `<257>` → `<256>e`：`<256>e cat in <256>e hat`
- 替换 `<256>` → "th"：`the cat in the hat`

&nbsp;
## 2. 一个简单的 BPE 实现

- 下面是上述算法的一个 Python 类实现，它模仿了 `tiktoken` 的 Python 用户界面
- 请注意，上面的编码部分通过 `train()` 描述了原始训练步骤；然而，`encode()` 方法的工作方式类似（尽管由于特殊 token 的处理，看起来有点复杂）：

1. 将输入文本拆分为单个字节
2. 当相邻 token（对）匹配已学习的 BPE 合并中的任何对时，反复查找并替换（合并）这些相邻 token（从最高到最低"等级"，即按学习顺序）
3. 继续合并，直到无法再应用任何合并
4. 最终的 token ID 列表就是编码输出

In [ ]:
from collections import Counter, deque
from functools import lru_cache


class BPETokenizerSimple:
    def __init__(self):
        # Maps token_id to token_str (e.g., {11246: "some"})
        self.vocab = {}
        # Maps token_str to token_id (e.g., {"some": 11246})
        self.inverse_vocab = {}
        # Dictionary of BPE merges: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Preprocess: Replace spaces with 'Ġ'
        # Note that Ġ is a particularity of the GPT-2 BPE implementation
        # E.g., "Hello world" might be tokenized as ["Hello", "Ġworld"]
        # (GPT-4 BPE would tokenize it as ["Hello", " world"])
        processed_text = []
        for i, char in enumerate(text):
            if char == " " and i != 0:
                processed_text.append("Ġ")
            if char != " ":
                processed_text.append(char)
        processed_text = "".join(processed_text)

        # Initialize vocab with unique characters, including 'Ġ' if present
        # Start with the first 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]

        # Extend unique_chars with characters from processed_text that are not already included
        unique_chars.extend(char for char in sorted(set(processed_text)) if char not in unique_chars)

        # Optionally, ensure 'Ġ' is included if it is relevant to your text processing
        if 'Ġ' not in unique_chars:
            unique_chars.append('Ġ')

        # Now create the vocab and inverse vocab dictionaries
        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # Add allowed special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # Tokenize the processed_text into token IDs
        token_ids = [self.inverse_vocab[char] for char in processed_text]

        # BPE steps 1-3: Repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            if len(token_ids) < 2:
                break
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None:  # No more pairs to merge. Stopping training.
                break
            
            updated = self.replace_pair(token_ids, pair_id, new_id)
            if updated == token_ids:
                break

            token_ids = updated
            self.bpe_merges[pair_id] = new_id

        # Build the vocabulary with merged tokens
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def encode(self, text):
        """
        Encode the input text into a list of token IDs.

        Args:
            text (str): The text to encode.

        Returns:
            List[int]: The list of token IDs.
        """
        tokens = []
        # Split text into tokens, keeping newlines intact
        words = text.replace("\n", " \n ").split()  # Ensure '\n' is treated as a separate token

        for i, word in enumerate(words):
            if i > 0 and not word.startswith("\n"):
                tokens.append("Ġ" + word)  # Add 'Ġ' to words that follow a space or newline
            else:
                tokens.append(word)  # Handle first word or standalone '\n'

        token_ids = []
        for token in tokens:
            if token in self.inverse_vocab:
                # token is contained in the vocabulary as is
                token_id = self.inverse_vocab[token]
                token_ids.append(token_id)
            else:
                # Attempt to handle subword tokenization via BPE
                sub_token_ids = self.tokenize_with_bpe(token)
                token_ids.extend(sub_token_ids)

        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Tokenize the token into individual characters (as initial token IDs)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        can_merge = True
        while can_merge and len(token_ids) > 1:
            can_merge = False
            new_tokens = []
            i = 0
            while i < len(token_ids) - 1:
                pair = (token_ids[i], token_ids[i + 1])
                if pair in self.bpe_merges:
                    merged_token_id = self.bpe_merges[pair]
                    new_tokens.append(merged_token_id)
                    # Uncomment for educational purposes:
                    # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                    i += 2  # Skip the next token as it's merged
                    can_merge = True
                else:
                    new_tokens.append(token_ids[i])
                    i += 1
            if i < len(token_ids):
                new_tokens.append(token_ids[i])
            token_ids = new_tokens

        return token_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        decoded_string = ""
        for token_id in token_ids:
            if token_id not in self.vocab:
                raise ValueError(f"Token ID {token_id} not found in vocab.")
            token = self.vocab[token_id]
            if token.startswith("Ġ"):
                # Replace 'Ġ' with a space
                decoded_string += " " + token[1:]
            else:
                decoded_string += token
        return decoded_string

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def find_freq_pair(token_ids, mode="most"):
        if(len(token_ids) < 2):
            return None
        pairs = Counter(zip(token_ids, token_ids[1:]))
        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_ids, pair_id, new_id):
        dq = deque(token_ids)
        replaced = []

        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                # Remove the 2nd token of the pair, 1st was already removed
                dq.popleft()
            else:
                replaced.append(current)

        return replaced


### 短 token 序列的边界情况处理

BPE 合并需要相邻的 token 对。  
如果 token 序列少于 2 个元素，则不存在对，因此 `find_freq_pair` 返回 `None`，训练会优雅地停止。

In [21]:
tok = BPETokenizerSimple()

assert tok.find_freq_pair([]) is None
assert tok.find_freq_pair([42]) is None

tok.train("", vocab_size=270)
tok.train("H", vocab_size=270)
tok.train("He", vocab_size=270)

print("Edge-case checks passed.")

Edge-case checks passed.


- 上面的 `BPETokenizerSimple` 类中有很多代码，在本笔记本中详细讨论它超出了范围，但下一节提供了一个简短的使用概述，以便更好地理解类方法

## 3. BPE 实现演练

- 在实践中，我强烈建议使用 [tiktoken](https://github.com/openai/tiktoken)，因为我上面的实现专注于可读性和教育目的，而不是性能
- 然而，其用法与 tiktoken 大致相似，只是 tiktoken 没有训练方法
- 让我们通过下面的一些示例来看看上面我的 `BPETokenizerSimple` Python 代码是如何工作的（详细的代码讨论超出了本笔记本的范围）

### 3.1 训练、编码和解码

- 首先，让我们考虑一些示例文本作为我们的训练数据集：

In [5]:
import os
import urllib.request

if not os.path.exists("../01_main-chapter-code/the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "../01_main-chapter-code/the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()

- 接下来，让我们初始化并训练 BPE tokenizer，词汇表大小为 1,000
- 请注意，由于前面讨论的字节值，词汇表大小默认已经是 255，所以我们只"学习"745 个词汇表条目
- 作为比较，GPT-2 的词汇表是 50,257 个 token，GPT-4 的词汇表是 100,256 个 token（tiktoken 中的 `cl100k_base`），GPT-4o 使用 199,997 个 token（tiktoken 中的 `o200k_base`）；与我们的简单示例文本相比，它们都有大得多的训练集

In [6]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

- 你可能想要检查词汇表内容（但请注意它会创建一个很长的列表）

In [7]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

1000


- 这个词汇表是通过合并 742 次创建的（~ `1000 - len(range(0, 256))`）

In [8]:
print(len(tokenizer.bpe_merges))

742


- 这意味着前 256 个条目是单字符 token

- 接下来，让我们通过 `encode` 方法使用创建的合并规则来编码一些文本：

In [9]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [10]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 42
Number of token IDs: 20


- 从上面的长度可以看出，一个 42 字符的句子被编码为 20 个 token ID，与基于字符字节的编码相比，有效地将输入长度缩减了一半

- 请注意，词汇表本身在 `decode()` 方法中使用，这使我们能够将 token ID 映射回文本：

In [11]:
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [12]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.


- 遍历每个 token ID 可以让我们更好地了解 token ID 是如何通过词汇表解码的：

In [13]:
for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

424 -> Jack
256 ->  
654 -> em
531 -> br
302 -> ac
311 -> ed
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
841 ->  ar
116 -> t
287 ->  a
466 -> nd
256 ->  
326 -> li
972 -> fe
46 -> .


- 正如我们所见，大多数 token ID 代表 2 字符子词；这是因为训练数据文本非常短，重复的单词不多，而且我们使用了相对较小的词汇表大小

- 作为总结，调用 `decode(encode())` 应该能够重现任意输入文本：

In [14]:
tokenizer.decode(tokenizer.encode("This is some text."))

'This is some text.'

&nbsp;
# 4. 总结

- 就是这样！这就是 BPE 的简要工作原理，包含一个用于创建新 tokenizer 的训练方法
- 我希望这个简短的教程对教育目的有所帮助；如果你有任何问题，请随时在[这里](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a)发起新的讨论


**这是一个用于教育目的的非常朴素的实现。[bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) 笔记本包含了一个更复杂（但更难阅读）的实现，它与 tiktoken 的行为匹配。**